<a href="https://colab.research.google.com/github/eviemcmahan/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [99]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [100]:
df['revenue'] = df['qty'] * df['price'] # Create revenue column
print(f'total revenue: ${df['revenue'].sum():.2f}')
print(f'total units: {df['revenue'].count()}')

total revenue: $8520.00
total units: 400


The purchases in the dataframe represent $8,520 in total revenue. There are 400 units of revenue that are summed up in this answer.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [101]:
by_category = df.groupby('category')[['revenue']].sum() # Group revenue by category
by_category['revenue_share'] = round((by_category['revenue'] / by_category['revenue'].sum() * 100),2) # Add revenue share column
by_category = by_category.sort_values('revenue', ascending=False) # Sort by revenue
print(by_category)

          revenue  revenue_share
category                        
Food       4293.0          50.39
Merch      1771.5          20.79
Drink      1554.0          18.24
RainGear    901.5          10.58


Food brought in the most revenue, at \$4,293 and 50.39% of total revenue. On the other hand, RainGear brought in the least revenue, at \$901.50 and 10.58%. I used `groupby` and `sort_values` to arrive at my answer.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [102]:
by_vendor = df.groupby('vendor_id')[['revenue']].sum() # Group revenue by vendor
by_vendor['order_count'] = df.groupby('vendor_id')['revenue'].count() # Add order count column
by_vendor['average_order_rev'] = round(by_vendor['revenue'] / by_vendor['order_count'],2) # Determine average order revenue by vendor
by_vendor = by_vendor.sort_values('average_order_rev', ascending=False) # Sort by average order revenue
print(by_vendor)

           revenue  order_count  average_order_rev
vendor_id                                         
V-01        2124.0           94              22.60
V-18        2349.0          108              21.75
V-05        1914.0           93              20.58
V-10        2133.0          105              20.31


The vendor with the highest average order revenue is V-01 with an average revenue of $22.60 per order. All four vendors have a similar order count of between 93 and 105 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [103]:
print(f'merch revenue share: {by_category.loc['Merch','revenue_share']:.1f}%') # Locate the Merch revenue share value.

merch revenue share: 20.8%


Merch accounts for 20.8% of all revenue. It brings in the second largest amount of revenue behind food. I calculated this number in Q2, so I just needed to locate the value.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [104]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
baseline_rows = len(df) # Determine initial rows
baseline_revenue = df['revenue'].sum() # Determine initial total revenue
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f} total revenue')

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one') # Left merge on df
print(f'after:  {len(joined)} rows, ${joined['revenue'].sum():.2f} total revenue \n') # Print check that rows and revenue didn't change
print(f'missing vendor:\n{joined.loc[joined['vendor_name'].isna(),'vendor_id'].value_counts()}') # Determine the missing vendor and it's counts


before: 400 rows, $8520.00 total revenue
after:  400 rows, $8520.00 total revenue 

missing vendor:
vendor_id
V-18    108
Name: count, dtype: int64


In [105]:
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor name') # Fill in missing vendor name with unknown vendor
print(joined.head())

  vendor_id  category  qty  price  revenue          vendor_name
0      V-10     Drink    2   24.0     48.0      Cav Merch North
1      V-18  RainGear    1   12.0     12.0  Unknown vendor name
2      V-18     Drink    3    4.5     13.5  Unknown vendor name
3      V-10      Food    2   12.0     24.0      Cav Merch North
4      V-18     Drink    3    7.5     22.5  Unknown vendor name


**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18. There are 108 counts that don't have this vendor. I labeled it `Unknown vendor name` since we don't have the vendor name. That is more descriptive than keeping the value `NaN`.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [106]:
# TODO

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [108]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

*a) Next game, I would tell*